<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L01-agent-loop-from-scratch/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L01-agent-loop-from-scratch/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T07-L01-agent-loop-from-scratch/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T07-L01-agent-loop-from-scratch/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T07-L01 · The agent loop from scratch, and why reliability compounds

**You will build:** a hand-rolled JSON-Schema-style argument validator, a tool registry that
never crashes on a bad call, capped seeded retries, an agent loop with explicit stop
conditions, and the evaluation harness that measures how reliable the whole thing is.

**Time:** ~55 minutes · **Runs on:** a laptop CPU, no GPU, no download ·
**Prerequisites:** T00-L01-the-8gb-track.

There is no language model anywhere in this lesson. `ScriptedPolicy` (section 9) is a
seeded, deterministic stand-in that knows its own task's correct plan and substitutes a
scripted mistake at a rate YOU choose — because the point of this lesson is the harness
around the model, which fails or succeeds the same way whatever sits behind it.

By the end you will be able to:

1. Implement a JSON-Schema-style argument validator, by hand, and a tool registry that
   turns an unknown tool name or a malformed call into a returned error rather than an
   exception.
2. Implement capped, seeded exponential backoff with full jitter and a retry wrapper that
   retries only transient tool failures, never a malformed call.
3. Implement the agent loop itself, with a step budget, a cost budget and a stuck-loop
   detector as its explicit stop conditions.
4. Measure success rate against task length over a seeded suite of synthetic tasks, with a
   bootstrap confidence interval, a cost-per-success figure and a failure taxonomy.
5. Fit the measured success-rate-versus-length curve against p^n and measure how much each
   guardrail — validation, retries, budget — moves success rate and cost.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import random
import sys
import time
import traceback
from typing import Any, Callable, NamedTuple, Sequence

import numpy as np

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

SEED = 20260923          # this lesson's one fixed seed: every task, every fault, every retry
                          # delay and every bootstrap resample derives from it, so re-running
                          # prints the same numbers on any machine.

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("validate_args",),
    "exercise 2": ("call_tool",),
    "exercise 3": ("backoff_delay",),
    "exercise 4": ("call_tool_with_retries",),
    "exercise 5": ("is_stuck",),
    "exercise 6": ("budget_status",),
    "exercise 7": ("run_agent_loop",),
    "exercise 8": ("bootstrap_success_interval",),
    "exercise 9": ("fit_reliability",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (backoff_delay)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. What an agent loop is, and why this lesson has no language model in it

An agent is a loop around a model: the model proposes an action, a harness validates and
executes it, the result goes back to the model as an observation, and the loop stops on
success, on failure, or on running out of budget. That sentence has no product name in it on
purpose — it is true of a frontier LLM calling tools and it is true of the toy policy this
lesson uses instead, because **this lesson deliberately does not call a language model.**
`ScriptedPolicy` (section 9) is a seeded, deterministic stand-in: it knows the correct plan
for its own task and, at a rate you choose, substitutes a scripted mistake instead — a
malformed argument, an unknown tool name, a hallucinated answer, a repeated action. It is not
a simplified LLM and this lesson never pretends otherwise; it is a knob you can set to an
EXACT number, so that the one measurement this lesson is built around — how success
probability falls as a task gets longer — is a property of the harness you build, not of
which model happened to answer your prompt today. The harness in this file has the same shape
as the one around a production agent, whatever sits behind `next_action`.

## 2. The world: chains of lookups

Every task in this lesson is a chain: a `start_key`, then a sequence of `lookup(key)` calls
that each return the NEXT key to look up, until the last one returns an integer `value`. The
agent's job is to walk the chain and then call `submit(value)` with the number at the end.
Chain **length** (how many `lookup` calls a task needs) is the one variable this lesson
controls exactly, which is what makes "does success fall off with task length" a real
measurement rather than a guess.

Below is the domain, given to you: the two tools, the task generator, and a registry
constructor whose `lookup` tool fails with a genuinely transient error a configurable
fraction of the time — a stand-in for a flaky network call, independent of anything the
policy does.

In [ ]:
class TransientToolError(Exception):
    """Raised by a tool for a failure worth retrying (a timeout, a dropped connection).

    Never raised for a mistake the CALLER made — a missing argument is never transient, because
    retrying it can never succeed. See exercise 2.
    """


class ToolResult(NamedTuple):
    ok: bool
    output: Any
    error: str | None
    transient: bool     # only meaningful when ok is False: worth a retry, or not?


class ToolRegistry(NamedTuple):
    tools: dict[str, Callable[..., Any]]
    schemas: dict[str, dict]


# The documented JSON-Schema subset every tool's arguments are checked against (exercise 1).
# Only "type", "required" and "enum" are supported — see claims.yaml for where each comes from.
TOOL_SCHEMAS: dict[str, dict] = {
    "lookup": {"type": "object", "properties": {"key": {"type": "string"}},
              "required": ["key"]},
    "submit": {"type": "object", "properties": {"value": {"type": "integer"}},
              "required": ["value"]},
}


class ChainTask(NamedTuple):
    task_id: int
    start_key: str
    key_sequence: list[str]          # the correct keys, in order; key_sequence[0] == start_key
    chain: dict[str, Any]            # key -> next key (str), or the terminal value (int)
    target: int
    length: int                      # == len(key_sequence)


def generate_chain_task(length: int, seed: int) -> ChainTask:
    """One deterministic lookup chain of the given length. Given to you."""
    rng = random.Random(seed)
    keys = [f"k{i}_{rng.randrange(1_000_000):06d}" for i in range(length)]
    chain: dict[str, Any] = {keys[i]: keys[i + 1] for i in range(length - 1)}
    target = rng.randint(100, 999)
    chain[keys[-1]] = target
    return ChainTask(task_id=seed, start_key=keys[0], key_sequence=keys, chain=chain,
                     target=target, length=length)


def generate_task_suite(lengths: Sequence[int], n_per_length: int, seed: int) -> list[ChainTask]:
    """`n_per_length` tasks at each length in `lengths`, all deterministic given `seed`."""
    tasks, task_id = [], 0
    for length in lengths:
        for _ in range(n_per_length):
            tasks.append(generate_chain_task(length, seed=seed * 1000 + task_id))
            task_id += 1
    return tasks


def make_registry(chain: dict, target: int, transient_rate: float,
                  rng: random.Random) -> ToolRegistry:
    """A registry over ONE task's chain. `lookup` fails transiently at `transient_rate`,
    independent of which key was asked for; an unrecognised key is a NON-transient error (a
    real mistake, not a flaky tool). Given to you."""
    def lookup(key):
        if rng.random() < transient_rate:
            raise TransientToolError("lookup timed out")
        if key not in chain:
            raise ValueError(f"unknown key {key!r}")
        val = chain[key]
        return {"next_key": val} if isinstance(val, str) else {"value": val}

    def submit(value):
        return {"correct": value == target}

    return ToolRegistry(tools={"lookup": lookup, "submit": submit}, schemas=TOOL_SCHEMAS)


_demo_task = generate_chain_task(length=3, seed=1)
print(f"a length-3 task: start at {_demo_task.start_key!r}, target {_demo_task.target}")
print("chain:", _demo_task.chain)

## 3. Exercise 1 — `validate_args()`

A tool call arrives as a plain dict of arguments. Before anything touches the tool itself,
check it against that tool's schema: is every required argument present, is each one the
right type, and — where the schema says so — is it one of an allowed set of values? This is
the harness's first line of defence, and it is what lets exercise 2 turn a malformed call
into a returned error instead of a crash.

<details><summary>💡 Hint 1 — what to think about</summary>

Two separate loops: one over `schema["required"]` (is each name present in `args`?), one
over `args.items()` (for each argument actually given, does its schema entry's "type" match,
and if the schema entry has an "enum", is the value in it?). Collect every problem you find
into a list — do not return on the first one. And: in Python, `isinstance(True, int)` is
`True`. JSON has no such rule. What does that mean for checking `"type": "integer"`?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Map each JSON Schema type name to a Python type (or tuple of types for "number"). For
"integer" and "number" specifically, reject a `bool` value even though `isinstance` would
accept it — check `isinstance(value, bool)` first and treat that as a type mismatch. An
argument name your schema does not mention is not an error in this subset: only `type`,
`required` and `enum` are checked.
</details>

In [ ]:
_ARG_TYPES = {"string": str, "integer": int, "number": (int, float), "boolean": bool,
             "array": list, "object": dict}


def validate_args(schema: dict, args: dict) -> list[str]:
    """Check `args` against a documented subset of JSON Schema, by hand.

    `schema` is an object schema: `schema["properties"]` maps an argument name to
    `{"type": ..., "enum": [...]}` (enum optional), and `schema["required"]` lists the names
    that must be present. Supported "type" values: "string", "integer", "number", "boolean",
    "array", "object". A Python `bool` is never accepted for "integer" or "number", even
    though `isinstance(True, int)` is `True` in Python — JSON has no such rule, and accepting
    a stray `True`/`False` here would silently let it stand in for 1/0.

    Returns a list of human-readable error strings — every problem found, not just the first
    — or an empty list if `args` is valid. Never raises: a schema mismatch is data for the
    harness to hand back to the caller, not an exception to propagate.

    Example:
        >>> schema = {"type": "object", "properties": {"key": {"type": "string"}},
        ...           "required": ["key"]}
        >>> validate_args(schema, {"key": "abc"})
        []
        >>> validate_args(schema, {})
        ["missing required argument 'key'"]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_validate_args() -> None:
    schema = TOOL_SCHEMAS["lookup"]
    assert validate_args(schema, {"key": "a"}) == [], "a valid call must return an empty list"
    missing = validate_args(schema, {})
    assert missing == ["missing required argument 'key'"], (
        f"got {missing!r} — a missing required argument needs exactly this message"
    )
    wrong_type = validate_args(schema, {"key": 5})
    assert wrong_type and "type" in wrong_type[0], (
        f"got {wrong_type!r} — an int where 'key' needs a string must be flagged"
    )
    bool_as_int = validate_args(TOOL_SCHEMAS["submit"], {"value": True})
    assert bool_as_int, (
        "validate_args({'value': True}) against an 'integer' schema returned no error — "
        "isinstance(True, int) is True in Python, but a bool must still be rejected here"
    )
    multi = validate_args({"type": "object",
                           "properties": {"a": {"type": "string"}, "b": {"type": "integer"}},
                           "required": ["a", "b"]}, {"a": 1, "b": "x"})
    assert len(multi) == 2, (
        f"got {len(multi)} error(s) for two wrong-typed arguments, expected 2 — "
        "collect every problem, do not return after the first"
    )
    enum_schema = {"type": "object", "properties": {"mode": {"type": "string",
                  "enum": ["fast", "slow"]}}, "required": ["mode"]}
    assert validate_args(enum_schema, {"mode": "fast"}) == [], (
        "a value that IS in the schema's 'enum' list must pass with no errors"
    )
    bad_enum = validate_args(enum_schema, {"mode": "medium"})
    assert bad_enum and "one of" in bad_enum[0], (
        f"got {bad_enum!r} — a value outside an 'enum' list must be flagged"
    )


_try("exercise 1", _check_validate_args)

## 4. Exercise 2 — `call_tool()`

`validate_args` only checks shape. `call_tool` is the harness boundary itself: look the tool
up, validate its arguments, call it, and turn EVERYTHING that can go wrong — an unknown tool
name, a validation failure, an exception the tool itself raises — into a returned
`ToolResult`. Nothing here may ever raise out to its caller. That is the one property a
production agent harness cannot ship without: a broken tool call is an observation the model
gets to react to, not a crash that ends the process.

<details><summary>💡 Hint 1 — what to think about</summary>

Three distinct failure shapes, each with its own `ToolResult`: the name is not in
`registry.tools` at all; `validate_args` finds a problem; the tool itself raises. For the
third, `try`/`except` around the call — and a `TransientToolError` is not the same kind of
failure as everything else.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

With `validate=True` (the default): unknown tool → `transient=False`; validation errors →
`transient=False` (retrying a malformed call can never help); `TransientToolError` →
`transient=True`; any other exception → `transient=False`. With `validate=False`: skip the
validation step and call the tool directly; now EVERY exception (including
`TransientToolError`) is marked `transient=True`, because without a validator there is no
way left to tell a flaky tool from a malformed call.
</details>

In [ ]:
def call_tool(registry: ToolRegistry, name: str, args: dict, validate: bool = True) -> ToolResult:
    """Execute one tool call through `registry`, never letting an exception escape.

    Looks `name` up; if it is not registered, returns a `ToolResult` naming the unknown tool.
    If `validate` is True (the harness's default), checks `args` with `validate_args()` first
    and returns every error found, with `transient=False` — a malformed call can never
    succeed no matter how many times it is retried.

    If `validate` is False, argument checking is skipped and the call is made directly.
    Whatever exception the tool then raises — a genuine `TransientToolError` from a flaky
    tool, or a bare `TypeError` from a call this validator would have rejected — comes back
    with `transient=True`: with no validator to tell them apart, the harness must guess, and
    the conservative guess is to assume either MIGHT be worth retrying.

    A successful call returns `ToolResult(ok=True, output=<return value>, error=None,
    transient=False)`.

    Example:
        >>> reg = ToolRegistry(tools={"double": lambda n: n * 2},
        ...                    schemas={"double": {"type": "object",
        ...                             "properties": {"n": {"type": "integer"}},
        ...                             "required": ["n"]}})
        >>> call_tool(reg, "double", {"n": 3}).output
        6
        >>> call_tool(reg, "triple", {"n": 3}).ok
        False
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_call_tool() -> None:
    reg = make_registry(_demo_task.chain, _demo_task.target, transient_rate=0.0,
                        rng=random.Random(0))
    unknown = call_tool(reg, "not_a_tool", {})
    assert not unknown.ok and not unknown.transient and "unknown tool" in unknown.error, (
        f"an unknown tool name must return ok=False, transient=False, got {unknown!r}"
    )
    bad_args = call_tool(reg, "lookup", {})
    assert not bad_args.ok and not bad_args.transient, (
        f"a validation failure must be non-transient, got {bad_args!r} — retrying a call "
        "missing its required argument can never succeed"
    )
    good = call_tool(reg, "lookup", {"key": _demo_task.start_key})
    assert good.ok and good.output == {"next_key": _demo_task.key_sequence[1]}, (
        f"a valid lookup on the demo task's start key returned {good!r} — with validation "
        "passing, call the tool with the arguments as keywords and wrap its return value"
    )
    try:
        bad_key = call_tool(reg, "lookup", {"key": "not-in-the-chain"})
    except Exception as exc:
        raise AssertionError(f"call_tool let the tool's own {type(exc).__name__} escape — catch "
                             "every exception the tool raises and return it as a ToolResult")
    assert not bad_key.ok and not bad_key.transient, (
        f"a key that is not in the chain is a real mistake, not a flaky tool — got {bad_key!r}"
    )
    flaky = ToolRegistry(
        tools={"boom": lambda: (_ for _ in ()).throw(TransientToolError("timeout"))},
        schemas={"boom": {"type": "object", "properties": {}, "required": []}})
    transient = call_tool(flaky, "boom", {})
    assert transient.transient, f"a TransientToolError must set transient=True, got {transient!r}"
    unvalidated = call_tool(reg, "lookup", {}, validate=False)
    assert unvalidated.transient, (
        f"with validate=False, EVERY exception — including one validate_args would have "
        f"caught — must come back transient=True, got {unvalidated!r}"
    )


_try("exercise 2", _check_call_tool)

## 5. Exercise 3 — `backoff_delay()`

A transient failure is worth retrying, but retrying immediately, every time, at the same
rate every caller does it, is how a struggling service gets hammered right when it can least
afford it. Capped exponential backoff — wait longer after each failure, up to a ceiling — is
the usual fix, built into the retry behaviour of most AWS SDKs; adding *jitter* (randomising
the wait rather than only lengthening it) is what actually spreads retries out (see
claims.yaml). This lesson never sleeps for the delay
it computes — see section 6 — but the NUMBER still has to be right, and it still has to be
reproducible, which means it has to come from the `rng` you were given, never from the
`random` module directly.

<details><summary>💡 Hint 1 — what to think about</summary>

Two numbers: the ceiling (`base` doubled once per `attempt`, capped at `cap`), and the delay
actually used (drawn uniformly between 0 and that ceiling). Which method of a
`random.Random` object draws a float uniformly between two bounds? And since `attempt` is 0
on the FIRST retry, how many doublings has that first ceiling had?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Work out the ceiling first — the exponential value for this attempt, never allowed above
`cap` — and only then draw, once, with zero as the lower bound: full jitter randomises the
WHOLE interval, not just its upper part. Draw from the `rng` you were handed, never from the
`random` module's own functions, or two runs sharing a seed would stop agreeing.
</details>

In [ ]:
BASE_DELAY = 0.05    # seconds (simulated — nothing in this lesson actually sleeps)
CAP_DELAY = 2.0


def backoff_delay(attempt: int, base: float, cap: float, rng: random.Random) -> float:
    """Full-jitter capped exponential backoff. `attempt` is 0 for the FIRST retry.

    The capped exponential ceiling is `min(cap, base * 2**attempt)`; the delay actually
    returned is drawn uniformly from `[0, ceiling)` using `rng.uniform` — `rng`, never the
    `random` module directly, so that two calls sharing the same seeded `rng` (in the same
    state) always draw the same sequence.

    Example:
        >>> d = backoff_delay(0, base=0.1, cap=8.0, rng=random.Random(0))
        >>> 0.0 <= d < 0.1
        True
        >>> backoff_delay(10, base=0.1, cap=8.0, rng=random.Random(0)) < 8.0
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_backoff_delay() -> None:
    d0 = backoff_delay(0, base=1.0, cap=100.0, rng=random.Random(7))
    assert 0.0 <= d0 < 1.0, f"attempt 0, base 1.0: expected [0, 1.0), got {d0}"
    d_capped = backoff_delay(20, base=1.0, cap=5.0, rng=random.Random(7))
    assert 0.0 <= d_capped < 5.0, (
        f"attempt 20 with cap=5.0 gave {d_capped} — base * 2**20 must be CAPPED at 5.0, not "
        "used uncapped"
    )
    a = backoff_delay(3, base=0.2, cap=10.0, rng=random.Random(42))
    b = backoff_delay(3, base=0.2, cap=10.0, rng=random.Random(42))
    assert a == b, "the same seed must draw the same delay every time"
    random.seed(999)                       # perturb the GLOBAL random module's state
    c = backoff_delay(3, base=0.2, cap=10.0, rng=random.Random(42))
    assert c == a, (
        f"got {c} after perturbing the global random module's state (expected {a}) — "
        "backoff_delay must draw from its `rng` argument only, never from `random` directly"
    )
    shared = random.Random(11)
    for attempt in range(4):
        ceiling = 0.5 * 2 ** attempt
        draws = [backoff_delay(attempt, base=0.5, cap=100.0, rng=shared) for _ in range(400)]
        assert max(draws) < ceiling, (
            f"attempt {attempt} with base 0.5 drew {max(draws):.3f}, above its ceiling "
            f"{ceiling} — attempt 0 is the FIRST retry, so its ceiling is base itself, "
            "doubled once per attempt after that"
        )
        assert min(draws) < 0.1 * ceiling and max(draws) > 0.9 * ceiling, (
            f"attempt {attempt}: 400 draws only covered [{min(draws):.3f}, {max(draws):.3f}] "
            f"of [0, {ceiling}) — full jitter draws from the WHOLE interval, starting at 0"
        )


_try("exercise 3", _check_backoff_delay)

## 6. Exercise 4 — `call_tool_with_retries()`

Retry a `TransientToolError` — nothing else. Retrying a malformed call, or a call to a tool
that does not exist, spends budget on something that can never succeed, however many times
you try it. This is why exercise 1 and exercise 2 came first: retrying correctly depends on
already knowing which failures are worth it.

Nothing in this function sleeps. A real caller would await `backoff_delay()` seconds between
attempts; this lesson only needs the NUMBER, both because a graded lesson can never put a
real wait on a required path (its time budget would turn every retry-heavy test into a
timeout) and because the point being measured — how much a retry costs and how much it buys — is
about the ATTEMPT count, not the wall clock.

<details><summary>💡 Hint 1 — what to think about</summary>

A `while` loop: call, and if the result failed AND `result.transient` AND you have not used
up `max_retries` yet, back off and call again. `attempt` for `backoff_delay` starts at 0 on
the FIRST retry, not the first call.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Count retries, not calls: start at zero and make the first call. Carry on only while the
latest result failed, was marked transient, and you still have retries left. Before each
retry, draw one delay for that retry's index from the `rng` you were given and add it to a
running total. The attempt count you return includes the very first call, so it is one more
than the number of retries made. Hand `validate` on to every `call_tool`, unchanged.
</details>

In [ ]:
def call_tool_with_retries(registry: ToolRegistry, name: str, args: dict, max_retries: int,
                           rng: random.Random, validate: bool = True) -> tuple[ToolResult, int, float]:
    """Call a tool, retrying only a TRANSIENT failure, up to `max_retries` times.

    Returns `(result, attempts, total_backoff)`: `attempts` is the number of calls actually
    made (1 if the first call already succeeded or failed non-transiently; up to
    `max_retries + 1` otherwise); `total_backoff` is the sum of every `backoff_delay` drawn.

    Example:
        >>> reg = ToolRegistry(tools={"ok": lambda: 1},
        ...                    schemas={"ok": {"type": "object", "properties": {}, "required": []}})
        >>> result, attempts, backoff = call_tool_with_retries(reg, "ok", {}, 3, random.Random(0))
        >>> attempts, result.ok, backoff
        (1, True, 0.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_call_tool_with_retries() -> None:
    reg = make_registry(_demo_task.chain, _demo_task.target, transient_rate=0.0,
                        rng=random.Random(0))
    result, attempts, backoff = call_tool_with_retries(reg, "lookup", {}, max_retries=5,
                                                        rng=random.Random(0))
    assert attempts == 1 and backoff == 0.0, (
        f"a non-transient failure (missing argument) got {attempts} attempts and backoff "
        f"{backoff} — it must never be retried"
    )
    always_flaky = ToolRegistry(
        tools={"boom": lambda: (_ for _ in ()).throw(TransientToolError("down"))},
        schemas={"boom": {"type": "object", "properties": {}, "required": []}})
    result2, attempts2, backoff2 = call_tool_with_retries(always_flaky, "boom", {},
                                                          max_retries=3, rng=random.Random(1))
    assert attempts2 == 4, f"max_retries=3 must allow 4 total attempts, got {attempts2}"
    assert not result2.ok, "a tool that always fails must still return a failed result"
    r = random.Random(1)
    expected_backoff = sum(backoff_delay(i, BASE_DELAY, CAP_DELAY, r) for i in range(3))
    assert abs(backoff2 - expected_backoff) < 1e-12, (
        f"total_backoff was {backoff2}, expected {expected_backoff} — draw exactly one "
        "backoff_delay per retry, from the SAME rng, in order"
    )
    calls = {"n": 0}

    def flaky_once():
        calls["n"] += 1
        if calls["n"] == 1:
            raise TransientToolError("first call fails")
        return 42

    recovers = ToolRegistry(tools={"flaky": flaky_once},
                            schemas={"flaky": {"type": "object", "properties": {},
                                               "required": []}})
    result3, attempts3, _ = call_tool_with_retries(recovers, "flaky", {}, max_retries=3,
                                                    rng=random.Random(2))
    assert attempts3 == 2 and result3.ok and result3.output == 42, (
        f"a tool that fails once then succeeds should take 2 attempts and succeed, got "
        f"attempts={attempts3} ok={result3.ok}"
    )
    _, attempts4, _ = call_tool_with_retries(reg, "lookup", {}, max_retries=2,
                                             rng=random.Random(3), validate=False)
    assert attempts4 == 3, (
        f"with validate=False a call missing its argument comes back transient=True, so it "
        f"IS retried (3 attempts at max_retries=2), got {attempts4} — pass `validate` on to "
        "call_tool instead of dropping it"
    )


_try("exercise 4", _check_call_tool_with_retries)

## 7. Exercise 5 — `is_stuck()`

Nothing about validation, retries or budgets stops a policy from doing the SAME thing over
and over without making any progress — the "never-stopping" fault this lesson injects on
purpose (section 9). The harness cannot see why an action repeats; it can only see that it
did. `is_stuck` is that observation, turned into a stop condition, so a loop pattern ends in
a handful of steps instead of burning the whole budget to find out nothing changes.

<details><summary>💡 Hint 1 — what to think about</summary>

"The same action" means the same tool name AND the same arguments AND the same outcome
(`ok`). A dict cannot go into a set or serve as a dict key, and two equal dicts may have been
built in different key orders — what turns a dict into something hashable whose value does
not depend on insertion order? And which entries of `trace` are you comparing: the first
ones, or the most recent ones?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Fewer than `window` entries means there is nothing to compare yet. Otherwise look ONLY at
the last `window` entries — not the first ones, and not one more than `window`. Give each a
signature made of its tool name, its arguments as sorted key/value pairs, and its `ok` flag,
and ask whether every signature matches the first.
</details>

In [ ]:
def is_stuck(trace: list[dict], window: int) -> bool:
    """True if the LAST `window` steps of `trace` are all the same action with the same outcome.

    Each entry of `trace` has at least "tool" (str), "args" (dict) and "ok" (bool). Returns
    False whenever `len(trace) < window`.

    Example:
        >>> t = [{"tool": "lookup", "args": {"key": "a"}, "ok": False}] * 3
        >>> is_stuck(t, window=3)
        True
        >>> is_stuck(t, window=4)
        False
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_is_stuck() -> None:
    repeating = [{"tool": "lookup", "args": {"key": "a"}, "ok": False}] * 3
    assert is_stuck(repeating, window=3), "3 identical entries with window=3 must be stuck"
    assert not is_stuck(repeating, window=4), (
        "only 3 entries exist; window=4 has nothing to compare and must return False"
    )
    varying = [{"tool": "lookup", "args": {"key": "a"}, "ok": False},
              {"tool": "lookup", "args": {"key": "b"}, "ok": False},
              {"tool": "lookup", "args": {"key": "c"}, "ok": False}]
    assert not is_stuck(varying, window=3), (
        "three DIFFERENT lookups (different 'key' each time) must not be flagged as stuck"
    )
    key_order = [{"tool": "submit", "args": {"a": 1, "b": 2}, "ok": True},
                {"tool": "submit", "args": {"b": 2, "a": 1}, "ok": True}]
    assert is_stuck(key_order, window=2), (
        "the same arguments built in a different key order must still count as the same call"
    )
    different_outcome = [{"tool": "lookup", "args": {"key": "a"}, "ok": False},
                        {"tool": "lookup", "args": {"key": "a"}, "ok": True}]
    assert not is_stuck(different_outcome, window=2), (
        "the same call that failed once and then succeeded made real progress — not stuck"
    )
    assert not is_stuck([], window=1), "an empty trace has nothing to compare and is not stuck"
    a_ok = {"tool": "lookup", "args": {"key": "a"}, "ok": True}
    b_fail = {"tool": "lookup", "args": {"key": "b"}, "ok": False}
    assert is_stuck([a_ok, b_fail, b_fail, b_fail], window=3), (
        "the last 3 entries are identical, so window=3 is stuck — an EARLIER, different entry "
        "must not hide a repeat at the end: compare exactly the last `window` entries"
    )
    assert not is_stuck([b_fail, b_fail, b_fail, a_ok], window=3), (
        "the repeat here is at the START and the latest entry differs, so this is not stuck — "
        "look at the most recent `window` entries, not the first ones"
    )


_try("exercise 5", _check_is_stuck)

## 8. Exercise 6 — `budget_status()`

Two independent budgets. **Step budget**: how many decisions the policy gets to make.
**Cost budget**: how many tool-call ATTEMPTS the episode may spend in total, retries
included — so a task can run out of cost while well inside its step budget, purely because
retries kept consuming attempts without the policy taking another turn. Exercise 7's check
builds exactly that case: one decision whose retries alone exhaust the cost budget.

<details><summary>💡 Hint 1 — what to think about</summary>

`step` counts decisions ALREADY made. If `max_steps` is 5, decisions 0, 1, 2, 3 and 4 are
allowed — which comparison against `max_steps` correctly refuses a 6th?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Two checks in a fixed order, each comparing a count already spent against its limit, where
reaching the limit exactly counts as exhausted. Step comes first, so that when both run out
on the same call the step budget is the one reported; if neither has run out, there is
nothing to report.
</details>

In [ ]:
def budget_status(step: int, cost_spent: int, max_steps: int, max_cost: int) -> str | None:
    """Which budget, if any, is exhausted — checked in this fixed order: step, then cost.

    `step` is the number of decisions already made; the step budget is exhausted once
    `step >= max_steps`. `cost_spent` is the number of tool-call attempts already spent; the
    cost budget is exhausted once `cost_spent >= max_cost`. Returns `"step_budget"`,
    `"cost_budget"`, or `None`.

    Example:
        >>> budget_status(step=4, cost_spent=1, max_steps=5, max_cost=100) is None
        True
        >>> budget_status(step=5, cost_spent=1, max_steps=5, max_cost=100)
        'step_budget'
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_budget_status() -> None:
    assert budget_status(0, 0, max_steps=5, max_cost=10) is None, (
        "with nothing spent yet, neither budget is exhausted — return None"
    )
    assert budget_status(4, 3, max_steps=5, max_cost=10) is None, (
        "step=4 with max_steps=5 must still have room for one more decision"
    )
    assert budget_status(5, 0, max_steps=5, max_cost=10) == "step_budget", (
        "step=5 with max_steps=5 must be exhausted — a boundary of exactly max_steps stops"
    )
    assert budget_status(0, 9, max_steps=5, max_cost=10) is None, (
        "cost_spent=9 with max_cost=10 must still have room"
    )
    assert budget_status(0, 10, max_steps=5, max_cost=10) == "cost_budget", (
        "cost_spent=10 with max_cost=10 must be exhausted"
    )
    assert budget_status(5, 10, max_steps=5, max_cost=10) == "step_budget", (
        "when BOTH budgets are exhausted on the same call, step_budget must win"
    )


_try("exercise 6", _check_budget_status)

## 9. The policy: a seeded stand-in, not a language model

`ScriptedPolicy` knows its own task's correct plan — an oracle, not intelligence — and, at
`fault_rate` per decision, substitutes one of four scripted mistakes instead: a malformed
argument, an unknown tool name, a hallucinated (premature, wrong) submission, or repeating
its own last action. Its `cursor` only advances when the LAST recorded action was exactly
the expected next step AND it succeeded — a fault, or a lucky-but-irrelevant success, never
moves it forward. Given to you: this is infrastructure, not the graded material.

In [ ]:
FAULT_KINDS = ("malformed_args", "unknown_tool", "hallucinated_submit", "repeat_last")


class AgentTrace(NamedTuple):
    steps: list[dict]
    status: str          # "success" | "wrong_answer" | "step_budget" | "cost_budget" | "stuck"
    cost_spent: int
    n_steps: int


class ScriptedPolicy:
    """A deterministic, seeded stand-in for "the model" that proposes actions.

    `plan` is the correct sequence of ("lookup", {"key": ...}) actions for this task;
    `final_action` is ("submit", {"value": target}). At each `next_action` call, with
    probability `fault_rate` a scripted mistake replaces the correct action instead.
    """

    def __init__(self, plan: list[tuple[str, dict]], final_action: tuple[str, dict],
                fault_rate: float, rng: random.Random):
        self.plan = list(plan)
        self.final_action = final_action
        self.fault_rate = fault_rate
        self.rng = rng
        self.cursor = 0

    def _expected(self) -> tuple[str, dict]:
        return self.plan[self.cursor] if self.cursor < len(self.plan) else self.final_action

    def next_action(self, trace: list[dict]) -> tuple[str, dict]:
        if trace:
            last = trace[-1]
            if (self.cursor < len(self.plan) and last["ok"] and last["tool"] == "lookup"
                    and last["args"] == self.plan[self.cursor][1]):
                self.cursor += 1
        correct_tool, correct_args = self._expected()
        if self.rng.random() >= self.fault_rate:
            return correct_tool, dict(correct_args)
        fault = self.rng.choice(FAULT_KINDS)
        target = self.final_action[1]["value"]
        if fault == "malformed_args":
            return ("lookup", {}) if correct_tool == "lookup" else ("submit", {"value": str(target)})
        if fault == "unknown_tool":
            return "look_up", dict(correct_args)
        if fault == "hallucinated_submit":
            guess = target
            while guess == target:
                guess = self.rng.randint(0, 999)
            return "submit", {"value": guess}
        # "repeat_last"
        if trace:
            return trace[-1]["tool"], dict(trace[-1]["args"])
        return "lookup", {}     # nothing to repeat yet on the very first decision


def _show_policy_walk() -> None:
    reg = make_registry(_demo_task.chain, _demo_task.target, transient_rate=0.0,
                        rng=random.Random(0))
    plan = [("lookup", {"key": k}) for k in _demo_task.key_sequence]
    policy = ScriptedPolicy(plan, ("submit", {"value": _demo_task.target}), fault_rate=0.0,
                            rng=random.Random(0))
    trace: list[dict] = []
    for _ in range(_demo_task.length + 1):
        tool, args = policy.next_action(trace)
        result = call_tool(reg, tool, args)
        trace.append({"tool": tool, "args": args, "ok": result.ok, "error": result.error,
                     "output": result.output})
        print(f"  {tool}({args}) -> ok={result.ok} output={result.output}")


_try("policy demo", _show_policy_walk, needs=("exercise 2",))

## 10. Exercise 7 — `run_agent_loop()`

Everything above, composed into one episode. Each turn: check the budgets first; ask the
policy for an action; execute it through `call_tool_with_retries` (never `call_tool`
directly — every call this loop makes may be retried); record the step; if it was a
`submit` that came back `ok=True`, the episode ends there, correct or not; otherwise check
`is_stuck`.

<details><summary>💡 Hint 1 — what to think about</summary>

`cost_spent` increases by `attempts`, not by 1 — a retried call costs more than a clean one.
`step`, the value passed to `budget_status`, is `len(trace)` at the top of the loop, not a
separately incremented counter. A `submit` that FAILS structurally (a bad argument) is a
mistake to recover from, not a final answer — it falls through to the stuck check like any
other step.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Loop until something ends the episode. At the top of each turn ask `budget_status` about
the steps and cost so far, and stop if it names a budget. Otherwise take the policy's
action, run it through `call_tool_with_retries` (passing `validate` along), add its attempts
to the cost, and append a step with every field the docstring lists. A `submit` that came
back ok ends the episode, as a success or a wrong answer according to what the tool said;
anything else goes on to the stuck check.
</details>

In [ ]:
def run_agent_loop(policy: ScriptedPolicy, registry: ToolRegistry, max_steps: int,
                   max_cost: int, max_retries: int, stuck_window: int,
                   rng: random.Random, validate: bool = True) -> AgentTrace:
    """Run one episode: ask `policy` for actions, execute them through `registry`, and stop.

    Each turn, in this order: ask `budget_status` about `len(steps)` decisions and the
    `cost_spent` so far, and stop with its status if it names one; take
    `tool, args = policy.next_action(steps)`; run that through `call_tool_with_retries` (with
    `max_retries`, `rng` and `validate`) and add its attempt count to `cost_spent`; append one
    step dict with the keys "tool", "args", "ok", "error", "output" (the last three from the
    `ToolResult`) and "attempts". A "submit" whose result is ok ends the episode as "success"
    if `output["correct"]` is true, else "wrong_answer". Any other step is followed by
    `is_stuck(steps, stuck_window)`, which ends the episode as "stuck" when it fires.

    Returns `AgentTrace(steps, status, cost_spent, n_steps)`, where `n_steps == len(steps)`
    and `cost_spent` is the sum of every step's "attempts". Get this wrong and every number
    from section 11 onward is wrong, even if every exercise before it is right.

    Example:
        >>> task = generate_chain_task(length=1, seed=5)
        >>> reg = make_registry(task.chain, task.target, transient_rate=0.0,
        ...                     rng=random.Random(0))
        >>> policy = ScriptedPolicy([("lookup", {"key": task.start_key})],
        ...                         ("submit", {"value": task.target}), fault_rate=0.0,
        ...                         rng=random.Random(0))
        >>> t = run_agent_loop(policy, reg, max_steps=10, max_cost=10, max_retries=2,
        ...                    stuck_window=3, rng=random.Random(0))
        >>> t.status, t.n_steps, t.cost_spent
        ('success', 2, 2)
        >>> sorted(t.steps[0])
        ['args', 'attempts', 'error', 'ok', 'output', 'tool']
    """
    # YOUR CODE HERE
    raise NotImplementedError


class _FixedPolicy:
    """A policy that plays back a fixed script — used only to unit-test run_agent_loop's
    stop conditions in isolation from ScriptedPolicy's randomness. Given to you."""

    def __init__(self, script: list[tuple[str, dict]]):
        self.script = script

    def next_action(self, trace):
        i = min(len(trace), len(self.script) - 1)
        return self.script[i]


def _check_run_agent_loop() -> None:
    task = generate_chain_task(length=2, seed=5)
    reg = make_registry(task.chain, task.target, transient_rate=0.0, rng=random.Random(0))
    plan = [("lookup", {"key": k}) for k in task.key_sequence]
    good_policy = ScriptedPolicy(plan, ("submit", {"value": task.target}), fault_rate=0.0,
                                 rng=random.Random(0))
    trace = run_agent_loop(good_policy, reg, max_steps=10, max_cost=10, max_retries=2,
                           stuck_window=3, rng=random.Random(0))
    assert trace.status == "success" and trace.n_steps == 3, (
        f"a length-2 task with a perfect policy must succeed in exactly 3 steps "
        f"(2 lookups + submit), got status={trace.status} n_steps={trace.n_steps}"
    )
    wrong_script = _FixedPolicy([("submit", {"value": task.target + 1})])
    wrong_trace = run_agent_loop(wrong_script, reg, max_steps=10, max_cost=10, max_retries=0,
                                 stuck_window=3, rng=random.Random(0))
    assert wrong_trace.status == "wrong_answer" and wrong_trace.n_steps == 1, (
        f"an immediate incorrect submit must end the episode as wrong_answer in 1 step, got "
        f"{wrong_trace.status}, {wrong_trace.n_steps} steps"
    )
    never_submits = _FixedPolicy([("lookup", {"key": "not-a-real-key"})])
    step_capped = run_agent_loop(never_submits, reg, max_steps=3, max_cost=1000, max_retries=0,
                                 stuck_window=100, rng=random.Random(0))
    assert step_capped.status == "step_budget" and step_capped.n_steps == 3, (
        f"a policy that never submits, with max_steps=3, must stop at exactly 3 steps with "
        f"status step_budget, got {step_capped.status}, {step_capped.n_steps} steps"
    )
    flaky_reg = make_registry(task.chain, task.target, transient_rate=1.0, rng=random.Random(0))
    cost_capped = run_agent_loop(never_submits, flaky_reg, max_steps=1000, max_cost=3,
                                 max_retries=10, stuck_window=100, rng=random.Random(0))
    assert cost_capped.status == "cost_budget", (
        f"an always-transient tool with max_cost=3 must exhaust cost before it exhausts "
        f"steps, got {cost_capped.status}"
    )
    assert cost_capped.n_steps == 1, (
        f"cost_budget must be able to trip on the very FIRST decision when max_cost is this "
        f"tight (1 call + up to 10 retries all count against it before the 2nd decision "
        f"is even considered) — got n_steps={cost_capped.n_steps}"
    )
    stuck_trace = run_agent_loop(never_submits, reg, max_steps=1000, max_cost=1000,
                                 max_retries=0, stuck_window=3, rng=random.Random(0))
    assert stuck_trace.status == "stuck" and stuck_trace.n_steps == 3, (
        f"3 identical failing lookups in a row with stuck_window=3 must stop as stuck at "
        f"exactly 3 steps, got status={stuck_trace.status} n_steps={stuck_trace.n_steps}"
    )
    partial_submit = _FixedPolicy([("submit", {"value": "not-an-int"}),
                                  ("submit", {"value": task.target})])
    recovers = run_agent_loop(partial_submit, reg, max_steps=10, max_cost=10, max_retries=0,
                              stuck_window=5, rng=random.Random(0))
    assert recovers.status == "success" and recovers.n_steps == 2, (
        f"a malformed submit must NOT end the episode — the next turn can still succeed, got "
        f"status={recovers.status} n_steps={recovers.n_steps}"
    )
    just_enough = run_agent_loop(ScriptedPolicy(plan, ("submit", {"value": task.target}),
                                                fault_rate=0.0, rng=random.Random(0)),
                                 reg, max_steps=3, max_cost=3, max_retries=2, stuck_window=3,
                                 rng=random.Random(0))
    assert just_enough.status == "success", (
        f"a correct submit on the LAST allowed step (3 of max_steps=3) must count as a success, "
        f"got {just_enough.status} — check the budgets at the top of a turn, before acting, "
        "not after the submit has already been made"
    )
    zero = run_agent_loop(never_submits, reg, max_steps=0, max_cost=10, max_retries=0,
                          stuck_window=3, rng=random.Random(0))
    assert zero.status == "step_budget" and zero.n_steps == 0, (
        f"max_steps=0 allows no decision at all, got {zero.status} after {zero.n_steps} steps "
        "— ask budget_status before the policy's first action too"
    )
    fields = {"tool", "args", "ok", "error", "output", "attempts"}
    missing = sorted(fields - set(trace.steps[0])) if trace.steps else sorted(fields)
    assert not missing, (
        f"each step dict needs the keys {sorted(fields)}; the first step is missing {missing} "
        "— later cells in this notebook read them"
    )
    assert cost_capped.cost_spent == sum(s["attempts"] for s in cost_capped.steps), (
        f"cost_spent ({cost_capped.cost_spent}) must equal the sum of every step's attempts"
    )
    unvalidated = run_agent_loop(partial_submit, reg, max_steps=10, max_cost=10, max_retries=0,
                                 stuck_window=5, rng=random.Random(0), validate=False)
    assert unvalidated.status == "wrong_answer" and unvalidated.n_steps == 1, (
        f"with validate=False nothing stops the string submit reaching the tool, which grades "
        f"it wrong at step 1; got status={unvalidated.status} n_steps={unvalidated.n_steps} — "
        "pass `validate` on to call_tool_with_retries"
    )


_try("exercise 7", _check_run_agent_loop)

Five individual episodes, at a fault rate high enough to see mistakes happen, with the full
trace printed for the first one:

In [ ]:
def _show_five_episodes() -> None:
    rng = random.Random(SEED)
    for i in range(5):
        task = generate_chain_task(length=4, seed=SEED + i)
        reg = make_registry(task.chain, task.target, transient_rate=0.1, rng=rng)
        plan = [("lookup", {"key": k}) for k in task.key_sequence]
        policy = ScriptedPolicy(plan, ("submit", {"value": task.target}), fault_rate=0.15,
                               rng=rng)
        trace = run_agent_loop(policy, reg, max_steps=40, max_cost=80, max_retries=3,
                              stuck_window=3, rng=rng)
        print(f"task {i}: length={task.length}  status={trace.status:<12} "
             f"steps={trace.n_steps:<3} cost={trace.cost_spent}")
        if i == 0:
            for s in trace.steps:
                print(f"    {s['tool']}({s['args']}) -> ok={s['ok']} attempts={s['attempts']}"
                     f"  error={s['error']}")


_try("five episodes demo", _show_five_episodes, needs=("exercise 7",))

## 11. The evaluation harness

`evaluate_suite` runs `run_agent_loop` once per task in a suite, with a fresh, independently
seeded policy, registry and retry-rng for each one, and records the outcome. Given to you —
it is a thin composition of exercise 7, not new material.

In [ ]:
class EvalResult(NamedTuple):
    outcomes: list[bool]
    lengths: list[int]
    statuses: list[str]
    costs: list[int]
    n_steps: list[int]


def evaluate_suite(tasks: list[ChainTask], fault_rate: float, transient_rate: float,
                   max_steps: int, max_cost: int, max_retries: int, stuck_window: int,
                   validate: bool, seed: int) -> EvalResult:
    """Run `run_agent_loop` once per task in `tasks`. Given to you."""
    outcomes, lengths, statuses, costs, n_steps = [], [], [], [], []
    for i, task in enumerate(tasks):
        # One stream per (seed, task, role), each named by a string. random.Random(str) hashes
        # the string with SHA-512 (not with Python's per-process hash randomisation), so the
        # streams are identical on every machine and Python version, and no two (seed, task,
        # role) triples can ever share one -- seed s + 1 is a genuinely new draw, not seed s
        # shifted along by a task.
        tool_rng = random.Random(f"{seed}:{i}:tool")
        policy_rng = random.Random(f"{seed}:{i}:policy")
        retry_rng = random.Random(f"{seed}:{i}:retry")
        registry = make_registry(task.chain, task.target, transient_rate, tool_rng)
        plan = [("lookup", {"key": k}) for k in task.key_sequence]
        policy = ScriptedPolicy(plan, ("submit", {"value": task.target}), fault_rate, policy_rng)
        trace = run_agent_loop(policy, registry, max_steps, max_cost, max_retries, stuck_window,
                              retry_rng, validate=validate)
        outcomes.append(trace.status == "success")
        lengths.append(task.length)
        statuses.append(trace.status)
        costs.append(trace.cost_spent)
        n_steps.append(trace.n_steps)
    return EvalResult(outcomes, lengths, statuses, costs, n_steps)


def success_rate_by_length(result: EvalResult) -> dict[int, tuple[int, int]]:
    """length -> (successes, n). Given to you."""
    by_length: dict[int, list[int]] = {}
    for length, ok in zip(result.lengths, result.outcomes):
        by_length.setdefault(length, [0, 0])
        by_length[length][1] += 1
        by_length[length][0] += int(ok)
    return {length: tuple(v) for length, v in sorted(by_length.items())}


def failure_taxonomy(result: EvalResult) -> dict[str, int]:
    """Counts of each non-success status. Given to you."""
    counts: dict[str, int] = {}
    for status, ok in zip(result.statuses, result.outcomes):
        if not ok:
            counts[status] = counts.get(status, 0) + 1
    return dict(sorted(counts.items(), key=lambda kv: -kv[1]))


def cost_per_success(result: EvalResult) -> float:
    """Total cost across the WHOLE suite (successes and failures alike) divided by the number
    of successes — the true price of a success rate, including money spent on the runs that
    did not pay off. Given to you."""
    n_success = sum(result.outcomes)
    if n_success == 0:
        return float("inf")
    return sum(result.costs) / n_success


LENGTHS = [1, 2, 3, 4, 6, 8, 10, 14, 18, 24]
N_PER_LENGTH = 40
FAULT_RATE = 0.05
TRANSIENT_RATE = 0.12
MAX_RETRIES = 3
STUCK_WINDOW = 3
MAX_STEPS = 80
MAX_COST = 200

_SUITE = generate_task_suite(LENGTHS, N_PER_LENGTH, seed=SEED)
print(f"suite: {len(_SUITE)} tasks over {len(LENGTHS)} lengths "
     f"({LENGTHS[0]}..{LENGTHS[-1]}), {N_PER_LENGTH} tasks per length")

## 12. Exercise 8 — `bootstrap_success_interval()`

A success rate measured over a finite suite is one draw from a distribution, not the truth.
A percentile bootstrap turns the raw outcomes into an interval: resample the outcomes with
replacement, many times, and read off where the middle 95% of the resampled rates fall.

<details><summary>💡 Hint 1 — what to think about</summary>

One resample is `n` indices drawn with replacement from `0 .. n - 1`. The generator's
integer-drawing method can draw a whole grid of them — `n_boot` rows of `n` — in one call.
Once you have looked the outcomes up through that grid, which axis holds ONE resample?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

The point estimate is the plain mean of the ORIGINAL outcomes, never the mean of the
resamples. Average each row of the looked-up grid to get one resampled success rate per
row, then take the two percentiles that leave 2.5% in each tail — numpy's percentile
function counts in percent, not as a fraction.
</details>

In [ ]:
def bootstrap_success_interval(outcomes: Sequence[bool], n_boot: int,
                               seed: int) -> tuple[float, float, float]:
    """Percentile bootstrap confidence interval for a success rate.

    Resamples `outcomes` with replacement, the same size as `outcomes`, `n_boot` times, using
    `numpy.random.default_rng(seed)`. Returns `(point_estimate, lo, hi)`: the point estimate
    is the observed mean of `outcomes`; `lo`/`hi` are the 2.5th/97.5th percentiles of the
    resampled means — a 95% interval.

    Example:
        >>> point, lo, hi = bootstrap_success_interval([True] * 20, n_boot=500, seed=0)
        >>> point, lo, hi
        (1.0, 1.0, 1.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_bootstrap_success_interval() -> None:
    all_true = bootstrap_success_interval([True] * 30, n_boot=1000, seed=0)
    assert all_true == (1.0, 1.0, 1.0), (
        f"30/30 successes must give an exact point interval (1.0, 1.0, 1.0), got {all_true}"
    )
    all_false = bootstrap_success_interval([False] * 30, n_boot=1000, seed=0)
    assert all_false == (0.0, 0.0, 0.0), f"0/30 successes must give (0.0, 0.0, 0.0), got {all_false}"
    mixed = [True] * 15 + [False] * 15
    point, lo, hi = bootstrap_success_interval(mixed, n_boot=5000, seed=1)
    assert abs(point - 0.5) < 1e-9, f"point estimate must be the observed mean exactly, got {point}"
    assert lo < 0.5 < hi, f"a 95% interval around a 50% rate with n=30 must not be a single point"
    assert hi - lo < 0.6, f"interval width {hi - lo:.3f} is implausibly wide for n=30"
    again = bootstrap_success_interval(mixed, n_boot=5000, seed=1)
    assert again == (point, lo, hi), "the same outcomes and seed must reproduce exactly"
    _, lo2, hi2 = bootstrap_success_interval([True] * 200 + [False] * 200, n_boot=4000, seed=3)
    assert 0.09 < hi2 - lo2 < 0.11, (
        f"200 successes in 400 gave an interval {hi2 - lo2:.4f} wide; a 95% interval here is "
        f"about {2 * 1.96 * (0.25 / 400) ** 0.5:.3f} wide — average each resample along its "
        "own row, and take the 2.5th and 97.5th percentiles, not the 5th and 95th"
    )


_try("exercise 8", _check_bootstrap_success_interval)

The baseline suite, evaluated once, with its overall interval and failure taxonomy. Cached
in `_EVAL_CACHE` (`_get_baseline()`) so every later section reuses the SAME run rather than
re-evaluating the whole suite each time it wants a number from it.

In [ ]:
_EVAL_CACHE: dict[str, EvalResult] = {}


def _get_baseline() -> EvalResult:
    if "baseline" not in _EVAL_CACHE:
        _EVAL_CACHE["baseline"] = evaluate_suite(_SUITE, FAULT_RATE, TRANSIENT_RATE, MAX_STEPS,
                                                 MAX_COST, MAX_RETRIES, STUCK_WINDOW,
                                                 validate=True, seed=SEED)
    return _EVAL_CACHE["baseline"]


def _show_baseline() -> None:
    baseline = _get_baseline()
    point, lo, hi = bootstrap_success_interval(baseline.outcomes, n_boot=4000, seed=SEED)
    print(f"overall success rate: {100 * point:.1f}%  (95% CI [{100 * lo:.1f}%, {100 * hi:.1f}%])")
    print(f"cost per success: {cost_per_success(baseline):.2f} tool-call attempts")
    print("failure taxonomy:", failure_taxonomy(baseline))
    print("\nsuccess rate by task length:")
    for length, (s, n) in success_rate_by_length(baseline).items():
        print(f"  length {length:>3}: {s:>3}/{n:<3} = {100 * s / n:5.1f}%")


_try("baseline suite", _show_baseline, needs=("exercise 7", "exercise 8"))

## 13. Exercise 9 — `fit_reliability()`

The point this whole lesson has been building toward: if every step of a task carries the
same independent chance `p` of NOT hitting a failure the harness cannot recover from, then
`success_rate ≈ p ** n`, and the table above lets you recover `p`. Note what `p` is. It is
not how often the policy is right; it is how often a step gets through without ending the
task, and the cell after this exercise measures the gap between the two. METR asks the same
shape of question about real agents: for each model it fits success probability against
task length — a logistic curve over human task time, where this lesson fits `p ** n` over a
count of steps (see claims.yaml).

<details><summary>💡 Hint 1 — what to think about</summary>

`log(success_rate) = n * log(p)` is a straight LINE through the origin — no intercept term,
because a zero-length task succeeds with certainty by construction. A length whose measured
success rate is exactly 0.0 has an undefined log — what should happen to that point before
it reaches the fit?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Drop every length whose success rate is zero, and refuse (with `ValueError`) if fewer than
two remain. Take logs of the rates. The model has ONE unknown, the slope, so the design
matrix has one column — the lengths — and no column of ones. Solve for that slope by least
squares with numpy's linear-algebra least-squares routine, then undo the log to get `p`.
</details>

In [ ]:
def fit_reliability(lengths: Sequence[float], success_rates: Sequence[float]) -> float:
    """Fit `success_rate ≈ p ** n` (n = task length) and return `p`.

    Uses only points with `success_rate > 0`; needs at least two such points, else raises
    `ValueError`. Fits `log(success_rate) = n * log(p)` by least squares with NO intercept term
    (forced through the origin), since a zero-length task must succeed with probability 1 by
    construction.

    Example:
        >>> round(fit_reliability([1, 2, 3, 4], [0.9, 0.81, 0.729, 0.6561]), 6)
        0.9
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_fit_reliability() -> None:
    exact = fit_reliability([1, 2, 3, 4], [0.9, 0.81, 0.729, 0.6561])
    assert abs(exact - 0.9) < 1e-6, f"exact p=0.9 data must recover p=0.9, got {exact}"
    with_zero = fit_reliability([1, 2, 3, 20], [0.9, 0.81, 0.729, 0.0])
    assert abs(with_zero - 0.9) < 1e-6, (
        f"a length with a ZERO success rate carries no usable information and must be "
        f"dropped, not fit against log(0) — got {with_zero}"
    )
    try:
        fit_reliability([5], [0.5])
        raise AssertionError("only one usable point must raise ValueError, not return a number")
    except ValueError:
        pass
    try:
        flat = fit_reliability([1, 2], [0.5, 0.5])
    except ValueError:
        raise AssertionError("fit_reliability([1, 2], [0.5, 0.5]) raised ValueError — two "
                             "usable points are enough; raise only when FEWER than two remain")
    assert abs(flat - 0.5 ** 0.6) < 1e-9, (
        f"got p={flat:.4f} for a success rate flat at 50% over lengths 1 and 2 (two points "
        "are enough). A free intercept soaks up the 50% and calls the slope zero, p = 1.0: no "
        "decay at all. Forced through the origin, least squares gives p = 0.5 ** 0.6 here"
    )


_try("exercise 9", _check_fit_reliability)

Fit `p` from the measured table above, then compare the predicted curve `p ** n` against
what the suite actually measured, length by length — the residual is printed, not asserted:

In [ ]:
def _show_fit_reliability() -> None:
    baseline = _get_baseline()
    by_length = success_rate_by_length(baseline)
    lengths_arr = [length for length, (s, n) in by_length.items()]
    rates_arr = [s / n for length, (s, n) in by_length.items()]
    p_hat = fit_reliability(lengths_arr, rates_arr)
    p_right = 1 - FAULT_RATE
    p_survive = 1 - FAULT_RATE / len(FAULT_KINDS)
    taxonomy = failure_taxonomy(baseline)
    n_failed = sum(taxonomy.values())
    print(f"fitted per-step reliability p = {p_hat:.4f}")
    print(f"the policy is right on a decision with probability 1 - fault_rate = {p_right:.4f}; "
         f"the fitted p is {'above' if p_hat > p_right else 'not above'} that")
    print(f"why: of the {n_failed} failed tasks, {taxonomy.get('wrong_answer', 0)} ended on a "
         f"wrong answer. A malformed call, an unknown tool name and a repeated action come "
         f"back to the policy as an observation it can recover from on its next turn; a "
         f"hallucinated submit is well-formed, so nothing in the harness can refuse it")
    print(f"if that is the only fatal kind among the {len(FAULT_KINDS)} the policy picks from, "
         f"a decision survives with probability 1 - fault_rate / {len(FAULT_KINDS)} = "
         f"{p_survive:.4f}; the "
         f"fitted p is {'below' if p_hat < p_survive else 'not below'} that, because a "
         f"length-n task takes n lookups PLUS a submit, plus a turn for every recovered mistake")
    print(f"\n{'length':>6}  {'measured':>9}  {'p**n predicted':>15}  {'abs diff':>9}")
    max_resid = 0.0
    for length, rate in zip(lengths_arr, rates_arr):
        pred = p_hat ** length
        resid = abs(rate - pred)
        max_resid = max(max_resid, resid)
        print(f"{length:>6}  {100 * rate:8.1f}%  {100 * pred:14.1f}%  {100 * resid:8.1f}%")
    print(f"\nlargest |measured - predicted| across all lengths: {100 * max_resid:.1f} "
         f"percentage points")
    n_long, rate_long = lengths_arr[-1], rates_arr[-1]
    no_recovery = p_right ** (n_long + 1)
    print(f"at length {n_long}: were EVERY mistake fatal, a task needing {n_long + 1} decisions "
         f"would finish with probability {p_right:.2f} ** {n_long + 1} = "
         f"{100 * no_recovery:.1f}%; this harness measured {100 * rate_long:.1f}% "
         f"({100 * (rate_long - no_recovery):+.1f} points) — what turning a mistake into an "
         f"observation, instead of an ending, is worth")


_try("reliability fit demo", _show_fit_reliability,
    needs=("exercise 7", "exercise 8", "exercise 9"))

## 14. How much does each guardrail actually move success and cost?

Three ablations, each changing exactly one guardrail and re-running the SAME suite with the
SAME seed. Every number below is computed here, not asserted.

In [ ]:
def _show_ablations() -> None:
    baseline = _get_baseline()
    base_rate = sum(baseline.outcomes) / len(baseline.outcomes)

    def mean_steps(result: EvalResult) -> float:
        return sum(result.n_steps) / len(result.n_steps)

    print(f"--- retries: max_retries={MAX_RETRIES} (baseline) vs max_retries=0 ---")
    no_retries = evaluate_suite(_SUITE, FAULT_RATE, TRANSIENT_RATE, MAX_STEPS, MAX_COST,
                               max_retries=0, stuck_window=STUCK_WINDOW, validate=True, seed=SEED)
    noretry_rate = sum(no_retries.outcomes) / len(no_retries.outcomes)
    print(f"  success rate: {100 * base_rate:.1f}% (retries on) vs {100 * noretry_rate:.1f}% "
         f"(retries off) -- {100 * (base_rate - noretry_rate):+.1f} points")
    print(f"  decisions per task: {mean_steps(baseline):.2f} (retries on) vs "
         f"{mean_steps(no_retries):.2f} (retries off) -- without a retry, a timed-out lookup "
         f"costs the policy a whole extra turn, and every turn is another chance to make a "
         f"fatal mistake")
    print(f"  cost per success: {cost_per_success(baseline):.2f} (retries on) vs "
         f"{cost_per_success(no_retries):.2f} (retries off)")

    print("\n--- validation: on (baseline) vs off, same generous budget ---")
    no_validation = evaluate_suite(_SUITE, FAULT_RATE, TRANSIENT_RATE, MAX_STEPS, MAX_COST,
                                  MAX_RETRIES, STUCK_WINDOW, validate=False, seed=SEED)
    noval_rate = sum(no_validation.outcomes) / len(no_validation.outcomes)
    print(f"  success rate: {100 * base_rate:.1f}% (validation on) vs {100 * noval_rate:.1f}% "
         f"(validation off) -- {100 * (base_rate - noval_rate):+.1f} points")
    print(f"  wrong answers: {failure_taxonomy(baseline).get('wrong_answer', 0)} (validation on) "
         f"vs {failure_taxonomy(no_validation).get('wrong_answer', 0)} (validation off) -- with "
         f"no validator, a submit whose value is a string reaches the tool, compares unequal to "
         f"the integer target and is graded wrong, instead of coming back as a fixable error")
    print(f"  cost per success: {cost_per_success(baseline):.2f} (validation on) vs "
         f"{cost_per_success(no_validation):.2f} (validation off) -- and every malformed lookup "
         f"is now retried up to {MAX_RETRIES} times for nothing, because the harness can no "
         f"longer tell it apart from a flaky tool")

    tight_length, tight_steps = 10, 8
    min_decisions = tight_length + 1       # every lookup, then the submit
    print(f"\n--- budget: length-{tight_length} tasks, max_steps={tight_steps} vs "
         f"{MAX_STEPS} ---")
    tight_tasks = generate_task_suite([tight_length], n_per_length=N_PER_LENGTH, seed=SEED)
    tight = evaluate_suite(tight_tasks, FAULT_RATE, TRANSIENT_RATE, max_steps=tight_steps,
                          max_cost=MAX_COST, max_retries=MAX_RETRIES, stuck_window=STUCK_WINDOW,
                          validate=True, seed=SEED)
    generous = evaluate_suite(tight_tasks, FAULT_RATE, TRANSIENT_RATE, max_steps=MAX_STEPS,
                             max_cost=MAX_COST, max_retries=MAX_RETRIES,
                             stuck_window=STUCK_WINDOW, validate=True, seed=SEED)
    tight_rate = sum(tight.outcomes) / len(tight.outcomes)
    generous_rate = sum(generous.outcomes) / len(generous.outcomes)
    print(f"  a length-{tight_length} task needs at least {min_decisions} decisions "
         f"({tight_length} lookups and a submit) to reach the correct submit")
    verdict = ("too few to ever reach it" if tight_steps < min_decisions else "enough to reach it")
    print(f"  max_steps={tight_steps} ({verdict}): {100 * tight_rate:.1f}% success -- "
         f"{failure_taxonomy(tight)}")
    print(f"  max_steps={MAX_STEPS} (generous): {100 * generous_rate:.1f}% success -- "
         f"{failure_taxonomy(generous)}")
    print("  a too-tight budget looks exactly like an unreliable policy in a success-rate "
         "number alone -- the failure taxonomy is what tells them apart")

    print(f"\nrecap: retries are worth {100 * (base_rate - noretry_rate):+.1f} success points; "
         f"validation is worth {100 * (base_rate - noval_rate):+.1f} success points and "
         f"{cost_per_success(no_validation) - cost_per_success(baseline):+.2f} attempts per "
         f"success saved; a budget of {tight_steps} steps on a task that needs "
         f"{min_decisions} decisions costs {100 * (generous_rate - tight_rate):.1f} success "
         f"points against the same policy with {MAX_STEPS} steps")


_try("guardrail ablations", _show_ablations, needs=("exercise 7",))

## 15. Common mistakes

- **Accepting `True`/`False` as an integer.** `isinstance(True, int)` is `True` in Python;
  in JSON a boolean is its own primitive type, not a number. `validate_args` must reject a
  bool where an integer or number is expected (see claims.yaml's
  `rfc8259-json-booleans-are-not-numbers`).
- **Retrying a non-transient failure.** A missing argument, an unknown tool, a bad key —
  none of these can ever succeed no matter how many times they are retried. Only
  `TransientToolError` is worth a retry.
- **Drawing backoff delays from the `random` module instead of the seeded `rng` argument.**
  Breaks reproducibility outright: the SAME lesson, run twice, can print two different
  traces.
- **Fitting `fit_reliability` WITH an intercept.** A two-parameter `log(rate) = a + n*log(p)`
  fit lets a bad first data point silently bend the WHOLE curve; forcing it through the
  origin (`n=0 -> rate=1`) is not optional here, it is the constraint the model is built on.
- **Comparing `trace` entries by tool name alone in `is_stuck`.** Two different lookups
  (different `key` values) are not the same action — only IDENTICAL arguments and outcome
  count as "no progress".
- **Treating cost as a synonym for steps.** A single decision that gets retried 3 times
  costs 4 attempts and one step. Conflating the two hides exactly what section 14's retries
  ablation measures.

In [ ]:
print(f"isinstance(True, int) = {isinstance(True, int)}  "
     f"-- why validate_args must special-case bool for 'integer' and 'number'")
print(f"'5' == 5 is {'5' == 5}  "
     f"-- why a type check catches what an equality check silently would not")

## 16. Self-check

1. `call_tool` catches an exception a tool raises. What must it NEVER do with it?
   - (a) Turn it into a `ToolResult` with `ok=False`
   - (b) Let it propagate out of `call_tool` and crash the process
   - (c) Record an `error` message describing what went wrong
   - (d) Decide, from its type, whether it is worth retrying

2. With `validate=False`, `call_tool` marks BOTH a `TransientToolError` and a plain
   `TypeError` from a malformed call as `transient=True`. Why is that the SAFER of the two
   possible mistakes, rather than marking both `False`?
   - (a) It is not safer; it is simply easier to implement
   - (b) Marking everything non-retryable would silently give up on a genuinely flaky tool
     that could have succeeded on a later attempt; marking everything retryable only wastes
     a bounded number of extra attempts on the calls that were never going to succeed anyway
   - (c) `TypeError` and `TransientToolError` always represent the same underlying problem
   - (d) Retrying is always free, so the direction of the mistake does not matter

3. A policy is right on each decision independently with probability 0.95. On a task that
   needs 20 right decisions in a row, the WHOLE task succeeds with probability closest to:
   - (a) 95%
   - (b) 64%
   - (c) 36%
   - (d) 5%

4. A task's episode ends with `status == "cost_budget"` while `n_steps` is far below
   `max_steps`. What does that combination, on its own, tell you?
   - (a) The policy is unusually unreliable
   - (b) Nothing is wrong; this is the expected outcome for every task
   - (c) Retries consumed attempts faster than steps accumulated — a flaky tool or a too-low
     `max_cost` relative to `max_retries`, not necessarily a bad policy
   - (d) `is_stuck` has a bug

5. This lesson's policy errs on a fixed fraction of its decisions, yet its measured success
   rate falls far more slowly with length than `(1 - fault_rate) ** n`. Why, mostly?
   - (a) The harness turns a malformed call, an unknown tool name or a repeated action into
     an observation the policy recovers from on its next turn, so only a well-formed wrong
     answer ends the episode
   - (b) Retries make the policy itself more accurate
   - (c) The step budget stops long tasks before they can fail
   - (d) The bootstrap interval corrects the measured success rate upwards

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
             "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## What you built

`validate_args`, `call_tool` and `call_tool_with_retries` are the boundary between a policy
and the world: nothing that boundary lets through can crash the harness, and nothing it
retries can be a mistake that was never going to succeed. `run_agent_loop` is the harness
itself — three independent stop conditions, none of which required a smarter model to add.
`evaluate_suite`, `bootstrap_success_interval` and `fit_reliability` are the instrument that
turned "reliability compounds" from an assertion in this file's prose into a number this
file's own code measured.

The tool-registry shape (`ToolRegistry`, `TOOL_SCHEMAS`, `ToolResult`) and the trace format
each step of `run_agent_loop` appends are named plainly on purpose: put any transport you
like between a policy and these tools, and what changes is how the actions travel, not the
reliability engineering this lesson just measured.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_validate_args, ()),
        ("exercise 2", _check_call_tool, ()),
        ("exercise 3", _check_backoff_delay, ()),
        ("exercise 4", _check_call_tool_with_retries, ()),
        ("exercise 5", _check_is_stuck, ()),
        ("exercise 6", _check_budget_status, ()),
        ("exercise 7", _check_run_agent_loop, ()),
        ("exercise 8", _check_bootstrap_success_interval, ()),
        ("exercise 9", _check_fit_reliability, ()),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.